# Smart Home Energy Predictor 2026
**Google Colab Training Notebook**

Pipeline: Data Load → Feature Engineering → Train (XGBoost / RF / Ridge) → Log MLflow

> MLflow logs จะถูกบันทึกลง **Google Drive** โดยตรง (`sqlite:///MyDrive/SmartEnergy2026/runs/mlruns.db`) — ไม่ต้อง ngrok

## 1. ติดตั้ง Dependencies

In [ ]:
# ติดตั้ง dependencies ทั้งหมด
!pip install mlflow xgboost scikit-learn pandas numpy matplotlib optuna pyyaml -q
print('✅ Dependencies installed')

## 2. Mount Google Drive และ Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
import os

# --- กำหนด path หลัก ---
BASE_PATH = '/content/drive/MyDrive/SmartEnergy2026'  # เปลี่ยนตาม Drive ของคุณ
REPO_DIR  = '/content/SmartEnergy2026'

# Clone repo (ถ้ายังไม่มี)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kaiwamairu/SmartEnergy2026.git {REPO_DIR}
else:
    print('Repo already exists — pulling latest...')
    !cd {REPO_DIR} && git pull

# เพิ่ม repo เข้า Python path
import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'✅ Working directory: {os.getcwd()}')

## 3. ตั้งค่า MLflow Tracking URI

In [ ]:
import os

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MLflow tracking ผ่าน SQLite บน Google Drive
# ไม่ต้องใช้ ngrok หรือ server ภายนอก
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

os.makedirs(f'{BASE_PATH}/runs', exist_ok=True)
MLFLOW_TRACKING_URI = f'sqlite:///{BASE_PATH}/runs/mlruns.db'

os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_TRACKING_URI

import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f'✅ MLflow Tracking URI: {MLFLOW_TRACKING_URI}')

# ทดสอบการเชื่อมต่อ
try:
    client = mlflow.tracking.MlflowClient()
    exps = client.search_experiments()
    print(f'✅ MLflow พร้อมใช้งาน — พบ {len(exps)} experiments')
except Exception as e:
    print(f'❌ MLflow error: {e}')

## 4. ดาวน์โหลด Dataset (UCI Household Power Consumption)

In [ ]:
import os
import zipfile

DATA_DIR = f'{BASE_PATH}/data'
os.makedirs(DATA_DIR, exist_ok=True)

RAW_FILE = f'{DATA_DIR}/household_power_consumption.txt'
PROCESSED_FILE = f'{DATA_DIR}/processed_hourly.csv'

if not os.path.exists(RAW_FILE):
    print('กำลังดาวน์โหลด UCI dataset...')
    !wget -q -O /tmp/household.zip https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip
    with zipfile.ZipFile('/tmp/household.zip', 'r') as z:
        z.extractall(DATA_DIR)
    print('✅ Dataset downloaded')
else:
    print(f'✅ Dataset already exists: {RAW_FILE}')

## 5. Preprocess Data (Resample → Hourly)

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.data.data_loader import load_raw_data, resample_hourly

if not os.path.exists(PROCESSED_FILE):
    print('กำลัง preprocess ข้อมูล...')
    df_raw = load_raw_data(RAW_FILE)
    print(f'Raw data shape: {df_raw.shape} | Date range: {df_raw.index[0]} → {df_raw.index[-1]}')

    df_hourly = resample_hourly(df_raw)
    df_hourly.to_csv(PROCESSED_FILE)
    print(f'✅ Processed data saved: {PROCESSED_FILE}')
    print(f'Hourly shape: {df_hourly.shape}')
else:
    import pandas as pd
    df_hourly = pd.read_csv(PROCESSED_FILE, index_col=0, parse_dates=True)
    print(f'✅ Loaded existing processed data: {df_hourly.shape}')

print('\nSample data:')
df_hourly[['Global_active_power']].tail()

## 6. ปรับ Config Path ให้ชี้ไป Google Drive

In [ ]:
import yaml

CONFIG_DIR = f'{REPO_DIR}/configs'
BASE_CFG_PATH = f'{CONFIG_DIR}/base.yaml'

# โหลดและอัปเดต base_path ใน config ให้ตรงกับ Drive
with open(BASE_CFG_PATH, 'r') as f:
    base_cfg = yaml.safe_load(f)

base_cfg['data']['base_path'] = BASE_PATH
base_cfg['mlflow']['tracking_uri'] = MLFLOW_TRACKING_URI

# เขียน config ชั่วคราวสำหรับ run นี้
RUNTIME_CFG_PATH = '/tmp/base_runtime.yaml'
with open(RUNTIME_CFG_PATH, 'w') as f:
    yaml.dump(base_cfg, f)

print('✅ Config updated with Drive paths')
print(f"  base_path: {base_cfg['data']['base_path']}")
print(f"  tracking_uri: {base_cfg['mlflow']['tracking_uri']}")

## 7. เทรน Baseline — Ridge Regression

In [ ]:
from src.pipelines.train_pipeline import run_training

print('='*60)
print('Training: Ridge Regression (Baseline)')
print('='*60)

result_ridge = run_training(
    model_name='ridge',
    data_path=PROCESSED_FILE,
    tracking_uri=MLFLOW_TRACKING_URI,
    version=1,
    config_dir=CONFIG_DIR,
)

print(f"\n✅ Ridge done | Run ID: {result_ridge['run_id']}")

## 8. เทรน Random Forest

In [ ]:
print('='*60)
print('Training: Random Forest')
print('='*60)

result_rf = run_training(
    model_name='random_forest',
    data_path=PROCESSED_FILE,
    tracking_uri=MLFLOW_TRACKING_URI,
    version=1,
    config_dir=CONFIG_DIR,
)

print(f"\n✅ Random Forest done | Run ID: {result_rf['run_id']}")

## 9. เทรน XGBoost (Main Model)

In [ ]:
print('='*60)
print('Training: XGBoost Regressor')
print('='*60)

result_xgb = run_training(
    model_name='xgboost',
    data_path=PROCESSED_FILE,
    tracking_uri=MLFLOW_TRACKING_URI,
    version=1,
    config_dir=CONFIG_DIR,
)

print(f"\n✅ XGBoost done | Run ID: {result_xgb['run_id']}")

## 10. เปรียบเทียบผลลัพธ์ทุกโมเดล

In [ ]:
import pandas as pd

results = {
    'Ridge (Baseline)': result_ridge,
    'Random Forest': result_rf,
    'XGBoost': result_xgb,
}

rows = []
for name, r in results.items():
    m = r['test_metrics']
    rows.append({
        'Model': name,
        'RMSE': round(m['rmse'], 4),
        'MAE': round(m['mae'], 4),
        'R²': round(m['r2'], 4),
        'Pass Threshold': '✅' if r['passed_threshold'] else '❌',
        'Run ID': r['run_id'][:8] + '...',
    })

df_results = pd.DataFrame(rows).sort_values('RMSE')
print('\n' + '='*70)
print('MODEL COMPARISON — Test Set Results')
print('='*70)
print(df_results.to_string(index=False))
print(f'\nProduction threshold: RMSE < 0.15')

## 11. Register Best Model ใน MLflow Registry

In [ ]:
# หาโมเดลที่ดีที่สุดและผ่าน threshold
best = min(results.items(), key=lambda x: x[1]['test_metrics']['rmse'])
best_name, best_result = best

if best_result['passed_threshold']:
    MODEL_REGISTRY_NAME = 'SmartEnergyPredictor'
    model_uri = f"runs:/{best_result['run_id']}/model"

    registered = mlflow.register_model(model_uri, MODEL_REGISTRY_NAME)
    print(f'✅ Model registered: {MODEL_REGISTRY_NAME} v{registered.version}')
    print(f'   Best model: {best_name}')
    print(f'   Test RMSE: {best_result["test_metrics"]["rmse"]:.4f}')
    print(f'   Run ID: {best_result["run_id"]}')
else:
    print(f'⚠️  Best model ({best_name}) RMSE = {best_result["test_metrics"]["rmse"]:.4f} ≥ threshold 0.15')
    print('   ยังไม่ผ่านเกณฑ์ — ลอง Hyperparameter Tuning ใน Cell ถัดไป')

## 12. (Optional) Hyperparameter Tuning ด้วย Optuna

In [ ]:
import optuna
import numpy as np
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from src.utils.config_loader import load_config
from src.utils.seed_utils import set_global_seed
from src.data.data_loader import load_processed_data
from src.data.preprocessor import prepare_features, time_series_split, fit_scaler, apply_scaler

# โหลดข้อมูลและ features (ใช้ซ้ำจาก cell ก่อน)
cfg = load_config('xgboost', config_dir=CONFIG_DIR)
set_global_seed(cfg['project']['seed'])

df = load_processed_data(PROCESSED_FILE)
X, y = prepare_features(df, cfg)
X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(
    X, y, cfg['data']['test_size'], cfg['data']['val_size']
)
X_train_s, scaler = fit_scaler(X_train)
X_val_s = apply_scaler(X_val, scaler)

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'objective': 'reg:squarederror',
        'n_jobs': -1,
        'verbosity': 0,
    }
    model = XGBRegressor(**params, random_state=cfg['project']['seed'])
    model.fit(X_train_s, y_train)
    val_rmse = np.sqrt(mean_squared_error(y_val, model.predict(X_val_s)))
    return val_rmse

# บันทึก study ลง Drive เพื่อให้รันต่อได้ถ้า Colab ตัด
STUDY_DB = f'sqlite:///{BASE_PATH}/runs/optuna_xgb.db'
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(),
    storage=STUDY_DB,
    study_name='xgb_energy_tuning',
    load_if_exists=True,
)

N_TRIALS = 50  # เพิ่มได้ถึง 100
print(f'Starting Optuna search — {N_TRIALS} trials...')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ Best RMSE: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

# บันทึก best params
import json
best_params_path = f'{BASE_PATH}/runs/best_xgb_params.json'
with open(best_params_path, 'w') as f:
    json.dump({'best_value': study.best_value, 'best_params': study.best_params}, f, indent=2)
print(f'Saved to: {best_params_path}')

## 13. เทรน XGBoost ด้วย Best Params จาก Optuna

In [ ]:
import json, mlflow, mlflow.xgboost, tempfile
from xgboost import XGBRegressor
from src.utils.mlflow_helpers import setup_mlflow, make_run_name, log_pip_freeze, log_scaler_params
from src.utils.config_loader import load_config
from src.utils.seed_utils import set_global_seed
from src.data.data_loader import load_processed_data
from src.data.preprocessor import prepare_features, time_series_split, fit_scaler, apply_scaler
from src.pipelines.train_pipeline import compute_metrics, plot_actual_vs_pred, plot_feature_importance

# เช็ค X_test_s โดยเฉพาะ (Cell 12 สร้างแค่ X_train_s, X_val_s — ไม่มี X_test_s)
try:
    _ = X_test_s
except NameError:
    print('X_test_s ไม่พบ — โหลด data และสร้าง splits ใหม่...')
    cfg = load_config('xgboost', config_dir=CONFIG_DIR)
    set_global_seed(cfg['project']['seed'])
    df = load_processed_data(PROCESSED_FILE)
    X, y = prepare_features(df, cfg)
    X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(
        X, y, cfg['data']['test_size'], cfg['data']['val_size'])
    X_train_s, scaler = fit_scaler(X_train)
    X_val_s  = apply_scaler(X_val, scaler)
    X_test_s = apply_scaler(X_test, scaler)
    print('✅ สร้าง X_test_s สำเร็จ')

try:
    _ = cfg
except NameError:
    cfg = load_config('xgboost', config_dir=CONFIG_DIR)

try:
    _ = N_TRIALS
except NameError:
    N_TRIALS = 50

# โหลด best params จาก Optuna
with open(f'{BASE_PATH}/runs/best_xgb_params.json') as f:
    best_data = json.load(f)

best_params = best_data['best_params']
best_params.update({
    'objective': 'reg:squarederror',
    'n_jobs': -1,
    'verbosity': 1,
    'early_stopping_rounds': 30,
})

setup_mlflow(MLFLOW_TRACKING_URI, cfg['mlflow']['experiment_name'])
mlflow.xgboost.autolog(log_models=True, log_datasets=False)
run_name = make_run_name('xgb_tuned', 2)

with mlflow.start_run(run_name=run_name):
    mlflow.set_tags({'algo': 'xgboost_tuned', 'optuna_trials': N_TRIALS})
    log_scaler_params(scaler)

    model_tuned = XGBRegressor(**best_params, random_state=42)
    model_tuned.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=50)

    test_metrics = compute_metrics(y_test, model_tuned.predict(X_test_s))
    mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})

    print(f"\nTuned XGBoost — RMSE: {test_metrics['rmse']:.4f} | MAE: {test_metrics['mae']:.4f} | R²: {test_metrics['r2']:.4f}")

    with tempfile.TemporaryDirectory() as tmpdir:
        plot_actual_vs_pred(y_test, model_tuned.predict(X_test_s), run_name,
                            f'{tmpdir}/actual_vs_pred_plot.png')
        mlflow.log_artifact(f'{tmpdir}/actual_vs_pred_plot.png')
        plot_feature_importance(model_tuned, list(X.columns), f'{tmpdir}/feature_importance.png')
        mlflow.log_artifact(f'{tmpdir}/feature_importance.png')
        log_pip_freeze(tmpdir)

    tuned_run_id = mlflow.active_run().info.run_id

threshold = cfg['baseline']['rmse_threshold']
if test_metrics['rmse'] < threshold:
    print(f'✅ RMSE {test_metrics["rmse"]:.4f} < {threshold} → Register ได้')
    registered = mlflow.register_model(f'runs:/{tuned_run_id}/model', 'SmartEnergyPredictor')
    print(f'   Registered as version {registered.version}')
else:
    print(f'⚠️  RMSE {test_metrics["rmse"]:.4f} ≥ {threshold}')

print(f'\n✅ Cell 13 done | Run ID: {tuned_run_id[:8]}...')

---
## P2 — Scale & Analysis

## 14. Model Comparison — All Runs จาก MLflow

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name(cfg['mlflow']['experiment_name'])
runs = client.search_runs(exp.experiment_id, order_by=["metrics.test_rmse ASC"])

# สร้างตารางเปรียบเทียบ
rows = []
for r in runs:
    m = r.data.metrics
    if 'test_rmse' in m:
        rows.append({
            'Run Name': r.info.run_name,
            'RMSE':  round(m.get('test_rmse', 999), 4),
            'MAE':   round(m.get('test_mae', 999), 4),
            'R²':    round(m.get('test_r2', 0), 4),
            'Algorithm': r.data.tags.get('algo', 'unknown'),
        })

df_cmp = pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)
df_cmp.index += 1
print('='*65)
print('ALL RUNS — Sorted by Test RMSE (ascending)')
print('='*65)
print(df_cmp.to_string())
print(f'\nProduction threshold: RMSE < 0.15')
print(f'Best model: {df_cmp.iloc[0]["Run Name"]} — RMSE {df_cmp.iloc[0]["RMSE"]}')

# Bar chart เปรียบเทียบ RMSE
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['RMSE', 'MAE', 'R²']
colors = ['#e74c3c', '#f39c12', '#2ecc71']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.barh(df_cmp['Run Name'], df_cmp[metric], color=color, alpha=0.8)
    ax.set_title(f'Test {metric}', fontsize=13, fontweight='bold')
    ax.set_xlabel(metric)
    for bar, val in zip(bars, df_cmp[metric]):
        ax.text(bar.get_width() + 0.0001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    if metric == 'RMSE':
        ax.axvline(x=0.15, color='red', linestyle='--', alpha=0.5, label='Threshold 0.15')
        ax.legend(fontsize=8)

plt.suptitle('Model Comparison — Smart Energy Predictor 2026', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/runs/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved: model_comparison.png')

## 15. Feature Importance Analysis (Top 20)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from xgboost import XGBRegressor

# โหลด model_tuned — retrain ถ้าไม่มีใน memory (เร็วกว่าโหลดจาก Registry)
try:
    _ = model_tuned
    print('✅ ใช้ model_tuned จาก memory')
except NameError:
    print('model_tuned ไม่พบใน memory — retrain จาก best_params...')
    from src.utils.config_loader import load_config
    from src.utils.seed_utils import set_global_seed
    from src.data.data_loader import load_processed_data
    from src.data.preprocessor import prepare_features, time_series_split, fit_scaler, apply_scaler

    cfg = load_config('xgboost', config_dir=CONFIG_DIR)
    set_global_seed(cfg['project']['seed'])
    df = load_processed_data(PROCESSED_FILE)
    X, y = prepare_features(df, cfg)
    X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(
        X, y, cfg['data']['test_size'], cfg['data']['val_size'])
    X_train_s, scaler = fit_scaler(X_train)
    X_val_s  = apply_scaler(X_val, scaler)
    X_test_s = apply_scaler(X_test, scaler)

    with open(f'{BASE_PATH}/runs/best_xgb_params.json') as f:
        best_data = json.load(f)
    best_params = best_data['best_params']
    best_params.update({'objective': 'reg:squarederror', 'n_jobs': -1,
                        'verbosity': 0, 'early_stopping_rounds': 30})

    model_tuned = XGBRegressor(**best_params, random_state=42)
    model_tuned.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)
    print('✅ Retrain สำเร็จ')

# Feature Importance
feature_names = list(X.columns)
importance = model_tuned.feature_importances_

fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('\n' + '='*50)
print('TOP 20 FEATURES — XGBoost Tuned')
print('='*50)
print(fi_df.head(20).to_string(index=False))

top20 = fi_df.head(20).sort_values('Importance')
fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top20['Feature'], top20['Importance'], color='#3498db', alpha=0.85)
ax.set_title('Feature Importance — Top 20\nXGBoost Tuned Model', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, top20['Importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/runs/feature_importance_p2.png', dpi=120, bbox_inches='tight')
plt.show()

lag_imp  = fi_df[fi_df['Feature'].str.contains('lag')]['Importance'].sum()
roll_imp = fi_df[fi_df['Feature'].str.contains('roll')]['Importance'].sum()
time_imp = fi_df[fi_df['Feature'].str.contains('hour|day|month|weekend')]['Importance'].sum()

print(f'\nFeature Group Summary:')
print(f'  Lag features    : {lag_imp:.4f} ({lag_imp*100:.1f}%)')
print(f'  Rolling features: {roll_imp:.4f} ({roll_imp*100:.1f}%)')
print(f'  Time features   : {time_imp:.4f} ({time_imp*100:.1f}%)')
print('✅ Saved: feature_importance_p2.png')

## 16. Residual Analysis & Prediction Deep Dive

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ตรวจสอบตัวแปรที่จำเป็น
try:
    _ = model_tuned
except NameError:
    raise RuntimeError('กรุณารัน Cell 15 ก่อน เพื่อโหลด model_tuned')

y_pred_test = model_tuned.predict(X_test_s)
residuals   = np.array(y_test) - y_pred_test

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Residual Analysis — XGBoost Tuned Model', fontsize=14, fontweight='bold')

# 1. Actual vs Predicted
ax = axes[0, 0]
n = min(500, len(y_test))
ax.plot(np.array(y_test)[:n], label='Actual', alpha=0.8, linewidth=1)
ax.plot(y_pred_test[:n], label='Predicted', alpha=0.8, linewidth=1, linestyle='--')
ax.set_title('Actual vs Predicted (first 500h)')
ax.set_xlabel('Time Step (hours)')
ax.set_ylabel('Global Active Power (kW)')
ax.legend()

# 2. Residual over time
ax = axes[0, 1]
ax.plot(residuals[:n], color='#e74c3c', alpha=0.6, linewidth=0.8)
ax.axhline(y=0, color='black', linewidth=1, linestyle='--')
ax.fill_between(range(n), residuals[:n], 0, alpha=0.2, color='#e74c3c')
ax.set_title('Residuals over Time')
ax.set_xlabel('Time Step (hours)')
ax.set_ylabel('Residual (Actual - Predicted)')

# 3. Residual distribution
ax = axes[1, 0]
ax.hist(residuals, bins=60, color='#3498db', alpha=0.8, edgecolor='white')
ax.axvline(x=0, color='red', linewidth=1.5, linestyle='--')
ax.axvline(x=residuals.mean(), color='orange', linewidth=1.5,
           label=f'Mean: {residuals.mean():.4f}')
ax.set_title('Residual Distribution')
ax.set_xlabel('Residual')
ax.set_ylabel('Count')
ax.legend()

# 4. Predicted vs Residual scatter
ax = axes[1, 1]
ax.scatter(y_pred_test, residuals, alpha=0.2, s=5, color='#2ecc71')
ax.axhline(y=0, color='red', linewidth=1.5, linestyle='--')
ax.set_title('Predicted vs Residual')
ax.set_xlabel('Predicted Value')
ax.set_ylabel('Residual')

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/runs/residual_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Residual Stats:')
print(f'  Mean : {residuals.mean():.4f} (ใกล้ 0 = ดี)')
print(f'  Std  : {residuals.std():.4f}')
print(f'  Max  : {residuals.max():.4f}')
print(f'  Min  : {residuals.min():.4f}')
print('✅ Saved: residual_analysis.png')

## 17. Log P2 Artifacts → MLflow

In [ ]:
import mlflow

# ดึง tuned_run_id จาก Registry ถ้าไม่มีใน memory
try:
    _ = tuned_run_id
except NameError:
    client = mlflow.tracking.MlflowClient()
    versions = client.search_model_versions("name='SmartEnergyPredictor'")
    latest = sorted(versions, key=lambda v: int(v.version))[-1]
    tuned_run_id = latest.run_id
    print(f'✅ ดึง run_id จาก Registry: {tuned_run_id[:8]}...')

# Log P2 artifacts เข้า run ของ best model
with mlflow.start_run(run_id=tuned_run_id):
    mlflow.log_artifact(f'{BASE_PATH}/runs/model_comparison.png')
    mlflow.log_artifact(f'{BASE_PATH}/runs/feature_importance_p2.png')
    mlflow.log_artifact(f'{BASE_PATH}/runs/residual_analysis.png')
    mlflow.log_metrics({
        'fi_lag_total':     round(float(lag_imp), 4),
        'fi_rolling_total': round(float(roll_imp), 4),
        'fi_time_total':    round(float(time_imp), 4),
        'residual_mean':    round(float(residuals.mean()), 4),
        'residual_std':     round(float(residuals.std()), 4),
    })

print('✅ P2 artifacts logged to MLflow run:', tuned_run_id[:8])
print('\nP2 เสร็จสมบูรณ์!')
print('  ✅ Model Comparison Chart')
print('  ✅ Feature Importance Analysis')
print('  ✅ Residual Analysis')
print('\nพร้อมไป P3 — Promote model สู่ Production')

---
## P3 — Production

## 18. Promote Best Model → Production Stage

In [ ]:
import mlflow

client = mlflow.tracking.MlflowClient()
MODEL_NAME = 'SmartEnergyPredictor'

versions = client.search_model_versions(f"name='{MODEL_NAME}'")
versions = sorted(versions, key=lambda v: int(v.version))

print('='*60)
print(f'Model Registry: {MODEL_NAME}')
print('='*60)

# เปรียบเทียบ RMSE ของทุก version — เลือก RMSE ต่ำสุด ไม่ใช่ version ล่าสุด
best_version = None
best_rmse = float('inf')

for v in versions:
    run = client.get_run(v.run_id)
    rmse = run.data.metrics.get('test_rmse', float('inf'))
    algo = run.data.tags.get('algo', 'unknown')
    print(f'  v{v.version} | {algo:20} | RMSE: {rmse:.4f} | Run: {v.run_id[:8]}...')
    if rmse < best_rmse:
        best_rmse = rmse
        best_version = v

print(f'\nBest version: v{best_version.version} | RMSE: {best_rmse:.4f}')

# Archive ทุก version แล้ว promote best
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=best_version.version,
    stage='Production',
    archive_existing_versions=True,
)

PROD_VERSION = best_version.version
PROD_RUN_ID  = best_version.run_id

print(f'\n✅ {MODEL_NAME} v{PROD_VERSION} → Production')
prod_run = client.get_run(PROD_RUN_ID)
print(f'   Algorithm : {prod_run.data.tags.get("algo", "unknown")}')
print(f'   Test RMSE : {best_rmse:.4f}')

## 19. Inference Pipeline — Raw Input → Prediction

In [ ]:
import numpy as np
import pandas as pd
from src.data.preprocessor import (
    create_lag_features, create_rolling_features,
    create_time_features, apply_scaler
)

def predict_energy(df_hourly: pd.DataFrame, model, scaler, cfg: dict) -> pd.Series:
    """
    Preprocessing wrapper: Hourly DataFrame → Scaled Features → Prediction
    รับ raw hourly data แล้วทำทุกขั้นตอนจบในที่เดียว

    Args:
        df_hourly : hourly resampled DataFrame (ต้องมี Global_active_power)
        model     : trained XGBoost model
        scaler    : fitted StandardScaler
        cfg       : config dict

    Returns:
        pd.Series ของ predictions พร้อม datetime index
    """
    target_col      = cfg['data']['target_column']
    lag_features    = cfg['features']['lag_features']
    rolling_windows = cfg['features']['rolling_windows']

    df = create_time_features(df_hourly.copy())
    df = create_lag_features(df, target_col, lag_features)
    df = create_rolling_features(df, target_col, rolling_windows)
    df = df.dropna()

    feature_cols = [c for c in df.columns if c != target_col]
    X_raw = df[feature_cols]
    X_scaled = apply_scaler(X_raw, scaler)

    preds = model.predict(X_scaled)
    return pd.Series(preds, index=df.index, name='predicted_power_kW')


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ทดสอบ Inference ด้วย 72 ชั่วโมงล่าสุดจาก test set
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from src.data.data_loader import load_processed_data
from src.utils.config_loader import load_config

cfg_inf = load_config('xgboost', config_dir=CONFIG_DIR)
df_full = load_processed_data(PROCESSED_FILE)

# ใช้ข้อมูล 200 ชั่วโมงล่าสุด (รวม lag buffer)
df_sample = df_full.tail(200)

predictions = predict_energy(df_sample, model_tuned, scaler, cfg_inf)
actuals     = df_sample[cfg_inf['data']['target_column']].loc[predictions.index]

# คำนวณ metrics บน sample
from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mae  = mean_absolute_error(actuals, predictions)

print('='*55)
print('INFERENCE TEST — Last 72h of dataset')
print('='*55)
print(f'  Samples  : {len(predictions)}')
print(f'  RMSE     : {rmse:.4f}')
print(f'  MAE      : {mae:.4f}')
print(f'\nSample predictions (last 5):')
df_inf = pd.DataFrame({'actual': actuals, 'predicted': predictions})
df_inf['error'] = (df_inf['actual'] - df_inf['predicted']).abs()
print(df_inf.tail().to_string())

# Plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(actuals.values[-72:],     label='Actual',    linewidth=2)
ax.plot(predictions.values[-72:], label='Predicted', linewidth=2, linestyle='--')
ax.set_title('Inference Test — Last 72 Hours', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour')
ax.set_ylabel('Global Active Power (kW)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/runs/inference_test.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved: inference_test.png')

## 20. Update CLAUDE.md Status & Final Summary

In [ ]:
import mlflow

# Log inference artifacts เข้า Production run
with mlflow.start_run(run_id=PROD_RUN_ID):
    mlflow.log_artifact(f'{BASE_PATH}/runs/inference_test.png')
    mlflow.set_tag('stage', 'production')
    mlflow.set_tag('prod_version', str(PROD_VERSION))

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Final Summary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
client = mlflow.tracking.MlflowClient()
prod_run = client.get_run(PROD_RUN_ID)
m = prod_run.data.metrics

print('='*65)
print('PROJECT COMPLETE — Smart Home Energy Predictor 2026')
print('='*65)
print(f'\n  Model    : SmartEnergyPredictor v{PROD_VERSION} (Production)')
print(f'  Algorithm: XGBoost Tuned (Optuna)')
print(f'  Run ID   : {PROD_RUN_ID[:16]}...')
print(f'\n  ── Test Set Metrics ──')
print(f'  RMSE : {m.get("test_rmse", 0):.4f}  (threshold: < 0.15 ✅)')
print(f'  MAE  : {m.get("test_mae",  0):.4f}')
print(f'  R²   : {m.get("test_r2",   0):.4f}')
print(f'\n  ── MLflow Artifacts ──')
print(f'  actual_vs_pred_plot.png')
print(f'  feature_importance.png')
print(f'  model_comparison.png       [P2]')
print(f'  feature_importance_p2.png  [P2]')
print(f'  residual_analysis.png      [P2]')
print(f'  inference_test.png         [P3]')
print(f'\n  ── Status ──')
print(f'  [x] P0 — Foundation')
print(f'  [x] P1 — Quality (Baseline + MLflow + Registry)')
print(f'  [x] P2 — Scale (Optuna + Feature Importance + Residual Analysis)')
print(f'  [x] P3 — Production (Promote + Inference Pipeline)')
print('\n' + '='*65)
print('✅ All stages complete!')